# B6 (Partie 1) — Préparation des données MVTec AD

**Objectif** : charger le dataset `bottle`, visualiser saines vs défauts, mettre en place un pipeline d'augmentation Albumentations et justifier les choix.

Lancer ce notebook depuis le répertoire `DL/` : `cd DL && jupyter notebook`

## §0 — Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Modules réutilisables (src/ est au même niveau que ce notebook)
sys.path.insert(0, str(Path("src")))

from indusense.vision.dataset import load_train_val, load_test, load_masks
from indusense.vision.augment import build_pipeline, augment_batch

# ── Constantes ──────────────────────────────────────────────────────────────
BOTTLE_ROOT = Path("bottle")
VAL_RATIO   = 0.2
SEED        = 42

Path("figures").mkdir(exist_ok=True)
np.random.seed(SEED)

print("Dataset :", BOTTLE_ROOT.resolve())
print("Dossier figures :", Path('figures').resolve())

## §1 — Chargement des données

Split fixe reproductible : 80 % des `train/good` → entraînement, 20 % → validation.  
La validation servira à calibrer le seuil d'anomalie en partie 2.

In [ ]:
X_train, X_val = load_train_val(BOTTLE_ROOT, val_ratio=VAL_RATIO, seed=SEED)
X_test, y_test, test_classes = load_test(BOTTLE_ROOT)

print(f"Train saines   : {X_train.shape}  → {X_train.shape[0]} images")
print(f"Val saines     : {X_val.shape}    → {X_val.shape[0]} images")
print(f"Test total     : {X_test.shape}   → {X_test.shape[0]} images")
print(f"  dont saines  : {(y_test == 0).sum()}")
print(f"  dont défauts : {(y_test == 1).sum()}")
print()
defect_classes = sorted(set(c for c in test_classes if c != "good"))
for cls in defect_classes:
    n = test_classes.count(cls)
    print(f"  {cls:20s}: {n} images")
print()
print(f"Plage de valeurs : [{X_train.min():.3f}, {X_train.max():.3f}]  (attendu [0, 1])")
print(f"dtype            : {X_train.dtype}")

## §2 — Visualisation : saines vs défauts

Ligne 1 : 4 images saines tirées aléatoirement depuis `train/good`.  
Ligne 2 : 1 exemple par classe de défaut (broken_large / broken_small / contamination) + 1 saine de test.

In [ ]:
rng = np.random.default_rng(SEED)
test_arr = np.array(test_classes)

fig, axes = plt.subplots(2, 4, figsize=(12, 6.5))
fig.suptitle(
    "MVTec AD — bottle (128×128, normalisé [0, 1])",
    fontsize=13, fontweight="bold", y=1.01
)

# ── Ligne 1 : saines ────────────────────────────────────────────────────────
idx_sain = rng.choice(len(X_train), 4, replace=False)
for col, idx in enumerate(idx_sain):
    axes[0, col].imshow(X_train[idx])
    axes[0, col].set_title("train/good", fontsize=9, color="#1A7F37")
    axes[0, col].axis("off")

# ── Ligne 2 : 1 défaut par classe + 1 saine test ───────────────────────────
for col, cls in enumerate(defect_classes[:3]):
    first_idx = np.where(test_arr == cls)[0][0]
    axes[1, col].imshow(X_test[first_idx])
    axes[1, col].set_title(cls.replace("_", "\n"), fontsize=9, color="#CF4526")
    axes[1, col].axis("off")

good_test_idx = np.where(test_arr == "good")[0][0]
axes[1, 3].imshow(X_test[good_test_idx])
axes[1, 3].set_title("test/good", fontsize=9, color="#1A7F37")
axes[1, 3].axis("off")

# ── Labels lignes ───────────────────────────────────────────────────────────
fig.text(0.01, 0.75, "Saines",  va="center", rotation="vertical",
         fontsize=11, fontweight="bold", color="#1A7F37")
fig.text(0.01, 0.28, "Défauts", va="center", rotation="vertical",
         fontsize=11, fontweight="bold", color="#CF4526")

plt.tight_layout()
plt.savefig("figures/01_saines_vs_defauts.png", dpi=120, bbox_inches="tight")
plt.show()
print("Sauvegardée : figures/01_saines_vs_defauts.png")

## §3 — Pipeline d'augmentation (Albumentations)

Le pipeline est appliqué **uniquement aux images saines d'entraînement**, jamais aux images de validation ou de test.

In [ ]:
pipeline = build_pipeline()
print("Pipeline d'augmentation :")
print(pipeline)

## §4 — Visualisation : original vs versions augmentées

In [ ]:
N_AUG = 5
sample = X_train[0:1]                              # (1, 128, 128, 3)
np.random.seed(SEED)                               # reproductibilité
augmented = augment_batch(sample, pipeline, n_aug=N_AUG)  # (N_AUG, 128, 128, 3)

fig, axes = plt.subplots(1, N_AUG + 1, figsize=(14, 3))
fig.suptitle(
    "Augmentation — 1 originale + 5 versions augmentées (même image)",
    fontsize=12, fontweight="bold"
)

axes[0].imshow(sample[0])
axes[0].set_title("Original", fontsize=10, fontweight="bold", color="#1A7F37")
axes[0].axis("off")

for i, aug_img in enumerate(augmented):
    axes[i + 1].imshow(aug_img)
    axes[i + 1].set_title(f"Aug #{i + 1}", fontsize=9)
    axes[i + 1].axis("off")

plt.tight_layout()
plt.savefig("figures/02_augmentation.png", dpi=120, bbox_inches="tight")
plt.show()
print("Sauvegardée : figures/02_augmentation.png")

## §5 — Justification des transformations retenues

### Transformations retenues

| Transform | Paramètres | Justification |
|-----------|-----------|---------------|
| `HorizontalFlip` | p=0.5 | La bouteille est symétrique gauche/droite — le flip est réaliste |
| `Rotate` | ±15°, p=0.7 | Légère inclinaison de pose caméra ou de placement → élargit la notion de « normal » sans la dénaturer |
| `RandomBrightnessContrast` | ±15%, p=0.7 | Variations d'éclairage industriel entre sessions de capture |
| `HueSaturationValue` | faibles amplitudes, p=0.5 | Léger drift de couleur entre lots de production |
| `GaussNoise` | p=0.3 | Bruit capteur caméra — simulé à faible intensité |

### Transformations écartées

| Transform | Raison d'exclusion |
|-----------|--------------------|
| `VerticalFlip` | La bouteille a un haut et un bas — une bouteille retournée n'est pas « normale » |
| `ElasticTransform` / `GridDistortion` | Déformation irréaliste sur un objet rigide en verre — risque de rendre tolérant aux défauts de forme |
| `RandomCrop` / `Zoom` fort | Couper la bouteille produit des images hors-distribution (le modèle apprendrait à reconstruire des demi-bouteilles) |
| `Rotation > 30°` | Au-delà, la bouteille sort du cadre ou ressemble à un défaut de positionnement |

### Point de réflexion — objet vs texture

Sur une **texture** (ex. `tile`), des flips verticaux et des rotations à 90° sont légitimes : le carrelage est isotrope.  
Sur un **objet centré** comme `bottle`, chaque transformation géométrique forte élargit la notion de « normal » et risque de rendre le modèle tolérant à des défauts géométriques (déformation, corps cassé).  
Le choix ici est conservateur : on favorise la **sensibilité aux anomalies** (précision de la reconstruction) plutôt qu'une robustesse géométrique maximale.